# Tratamento de outliers no mínimo e no máximo
O tratamento da média (notebook `outliers`), copiado para as bases do mínimo e do máximo: as mesmas 3 funções, com as mesmas escolhas (os padrões delas, como na média):
- `detect_outliers_iqr` em escala log: a máscara de valores (**qual variável** é extrema);
- `detect_outliers_ensemble` (IF + LOF + KNN, por maioria): a marcação dos meses (**qual mês** é estranho);
- `tratar_outliers`: um valor vira NaN quando o IQR o marca **e** o ensemble marca o mês.

Antes, uma conferência: as mesmas 3 chamadas na média reproduzem a `dados_med_tratados.xlsx`. As bases gravadas aqui (`dados_min_tratados.xlsx` e `dados_max_tratados.xlsx`) são usadas no notebook `imputacoes_mesma_prova_min_max` (o apêndice do A).

As escolhas do tratamento (contaminações, regra do ensemble, IQR em log) foram feitas olhando a média e não foram revistas para o mínimo e o máximo: é uma limitação a citar no texto.

# Set Up

In [1]:
import pandas as pd
from pcj.utils import dados, BASE_DIR
from pcj.processamento import decompor_estatisticas
from pcj.outliers import detect_outliers_iqr, detect_outliers_ensemble, tratar_outliers

In [2]:
# A base limpa, com as 3 estatísticas (como no notebook outliers)
df, metadados, variaveis = dados(r"data\processed\dados_limpos.xlsx")

# Conferência na média
As 3 chamadas na média, como no notebook `outliers`. O resultado precisa ser igual à `dados_med_tratados.xlsx`: os mesmos valores e as mesmas células vazias.

In [3]:
df_med = decompor_estatisticas(df, estatistica="med")
_, _, mascara_med = detect_outliers_iqr(df_med, variaveis, log=True)
tabela_ens_med = detect_outliers_ensemble(df_med, variaveis)
med_tratada, registro_med = tratar_outliers(df_med, tabela_ens_med["ensemble"], mascara_med)

# A tratada gravada pelo notebook outliers. Os índices são diferentes (no Excel, de 0 a 190),
# por isso as duas são comparadas pela data (set_index) e só nas variáveis
med_excel, _, _ = dados(r"data\processed\dados_med_tratados.xlsx")
de_novo = med_tratada.set_index("Data")[variaveis]
gravada = med_excel.set_index("Data")[variaveis]

print("Média: meses marcados", tabela_ens_med["ensemble"].sum(), "| valores tratados", len(registro_med))
print("Maior diferença para o Excel:", (de_novo - gravada).abs().max().max(),
      "| as mesmas células vazias:", (de_novo.isna() == gravada.isna()).all().all())

Média: meses marcados 14 | valores tratados 32
Maior diferença para o Excel: 0.0 | as mesmas células vazias: True


# Mínimo e máximo
Para cada estatística: a máscara do IQR (log), o ensemble e o tratamento.

In [4]:
# Resultados de cada estatística, em dicionários com a chave "min" ou "max"
bases = {}
ensembles = {}
tratadas = {}
registros = {}

for estatistica in ["min", "max"]:
    base = decompor_estatisticas(df, estatistica=estatistica)

    # 1. Máscara de valores: o IQR em escala log (como na média)
    _, _, mascara = detect_outliers_iqr(base, variaveis, log=True)

    # 2. Marcação dos meses: o ensemble padrão (IF + LOF + KNN, por maioria)
    tabela_ens = detect_outliers_ensemble(base, variaveis)

    # 3. Tratamento: NaN nos valores marcados pelo IQR, só nos meses marcados pelo ensemble
    tratada, registro = tratar_outliers(base, tabela_ens["ensemble"], mascara)

    bases[estatistica] = base
    ensembles[estatistica] = tabela_ens
    tratadas[estatistica] = tratada
    registros[estatistica] = registro

    print(f"{estatistica}: meses marcados {tabela_ens['ensemble'].sum()} | valores tratados {len(registro)}")

min: meses marcados 14 | valores tratados 36
max: meses marcados 15 | valores tratados 32


## Meses marcados
Os meses que o ensemble marcou em pelo menos uma estatística (True = marcado). A última coluna conta em quantas das 3 estatísticas o mês foi marcado.

In [5]:
# As 3 bases têm os mesmos 191 meses, na mesma ordem (conferência abaixo); por isso, as marcações
# podem ir lado a lado, com a data como índice (.values pega só os valores, sem o índice de cada base)
print("Mesmos meses, na mesma ordem:",
      (bases["min"]["Data"].values == df_med["Data"].values).all() and (bases["max"]["Data"].values == df_med["Data"].values).all())

marcados = pd.DataFrame(index=df_med["Data"].dt.strftime("%Y-%m"))
marcados["media"] = tabela_ens_med["ensemble"].values
marcados["min"] = ensembles["min"]["ensemble"].values
marcados["max"] = ensembles["max"]["ensemble"].values

# Em quantas estatísticas o mês foi marcado (True vale 1); só os meses marcados em alguma
marcados["n_estatisticas"] = marcados[["media", "min", "max"]].sum(axis=1)
marcados[marcados["n_estatisticas"] > 0]

Mesmos meses, na mesma ordem: True


,media,min,max,n_estatisticas
Data,,,,
2009-02,False,True,False,1
2009-10,False,False,True,1
2009-12,True,False,True,2
2010-12,False,True,False,1
2012-09,False,True,False,1
2014-02,True,True,True,3
2014-04,True,True,False,2
2014-07,True,True,True,3
2014-08,True,True,True,3


## Valores tratados
O registro de cada estatística: uma linha por valor trocado por NaN (data, variável e valor original).

In [6]:
for estatistica in ["min", "max"]:
    print(estatistica)
    display(registros[estatistica])

min


,Data,variavel,valor_original
0,2009-02-01,COR,393.00
1,2009-02-01,pH,6.50
2,2009-02-01,Fe,6.82
3,2012-09-01,AC.,32.00
4,2014-02-01,ALC.,123.00
5,2014-02-01,Clorofila,509.64
6,2014-07-01,ALC.,154.00
7,2014-07-01,AC.,22.00
8,2014-08-01,ALC.,188.00
9,2014-08-01,AC.,40.00


max


,Data,variavel,valor_original
0,2014-02-01,pH,8.00
1,2014-02-01,Clorofila,509.64
2,2014-07-01,AC.,95.00
3,2014-07-01,Cl,192.00
4,2014-08-01,ALC.,201.00
5,2014-08-01,AC.,90.00
6,2014-08-01,O.D.,1.70
7,2014-09-01,pH,8.10
8,2014-09-01,ALC.,220.00
9,2014-09-01,O.C.,33.00


Valores tratados por variável, nas 3 estatísticas.

In [7]:
# value_counts conta as linhas de cada variável no registro; reindex põe as variáveis na ordem da base
# (as que não aparecem ficam com NaN, e o fillna(0) as troca por 0)
contagem = pd.DataFrame({
    "media": registro_med["variavel"].value_counts(),
    "min": registros["min"]["variavel"].value_counts(),
    "max": registros["max"]["variavel"].value_counts()
})
contagem = contagem.reindex(variaveis).fillna(0).astype(int)

# Linha nova "Total": a soma de cada coluna
contagem.loc["Total"] = contagem.sum()
contagem

,media,min,max
variavel,,,
COR,1,3,0
TURB.,0,2,0
pH,3,2,6
ALC.,4,4,3
AC.,3,4,3
O.C.,2,3,3
O.D.,1,3,3
Cl,6,4,4
DUR.,1,1,0


## Meses completos
Antes e depois do tratamento, em cada estatística. O A só sorteia falhas nos meses completos.

In [8]:
linhas = []
for estatistica, base, tratada in [("media", df_med, med_tratada), ("min", bases["min"], tratadas["min"]),
                                   ("max", bases["max"], tratadas["max"])]:
    linhas.append({
        "estatistica": estatistica,
        "completos_antes": len(base[variaveis].dropna()),
        "completos_depois": len(tratada[variaveis].dropna())
    })

pd.DataFrame(linhas)

,estatistica,completos_antes,completos_depois
0,media,126,118
1,min,126,118
2,max,125,118


# Gravação
`data/processed/dados_min_tratados.xlsx` e `dados_max_tratados.xlsx`, com as mesmas colunas da `dados_med_tratados.xlsx` (nome fixo, como ela). Rodar de novo dá os mesmos números: as seeds do ensemble são fixas.

In [9]:
tratadas["min"].to_excel(BASE_DIR / "data" / "processed" / "dados_min_tratados.xlsx", index=False)
tratadas["max"].to_excel(BASE_DIR / "data" / "processed" / "dados_max_tratados.xlsx", index=False)

print("Gravado em:", BASE_DIR / "data" / "processed")

Gravado em: C:\_Master\Programacao\Guilherme-Betta\1_Projects\IC2\IC2-item-7\data\processed
